# ***Notebook For Standardizing For Logistic DB Tables***

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
import os
from pathlib import Path

from sqlalchemy import create_engine, text

import warnings
warnings.filterwarnings('ignore')

In [2]:
env_path = Path.cwd()/"github_repo_cloned"/"the_distribution_hub_optimizer"/".env"
load_dotenv(env_path)


host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
paswd = os.getenv("DB_PASSWORD")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")
output_dir = Path(os.getenv("OUTPUT_DIR", Path.cwd()/"outputs"))
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
engine = create_engine(f"mysql+pymysql://{user}:{paswd}@{host}:{port}/{db}")


def sql(query, engine = engine):
    with engine.connect() as conn:
        df = pd.read_sql(text(query),conn)
    return df

In [4]:
bill = sql('select * from bill') 

In [5]:
gsdp = sql('select * from gsdp') 

In [6]:
population = sql('select * from population')

## **Data Standardization**

In [7]:
bill.head()

,state_code,state_name,year,month,intra_suplliers,intra_bill,intra_assets,inter_out_suplliers,inter_out_bill,inter_out_asset,inter_in_suplliers,inter_in_bill,inter_in_asset
0,1,JAMMU AND KASHMIR,2018,7,3171,120989,2083.7662087950002,1711,25744,2243.508837,15449,151999,4567.124618
1,1,JAMMU AND KASHMIR,2018,8,3292,126112,2245.7754784900003,1996,30376,2535.885482,17023,180173,5157.642880
2,1,JAMMU AND KASHMIR,2018,9,3484,130396,2387.847554812,2114,30149,2215.296839,16962,186069,5240.539348
3,1,JAMMU AND KASHMIR,2018,10,3784,135435,2545.633385386,2273,32807,2160.836647,17722,202717,5138.989032
4,1,JAMMU AND KASHMIR,2018,11,3851,123745,2178.749353559,2080,26704,1751.514975,16757,165151,4435.449635


In [8]:
gsdp.head()

,state/ut,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24
0,Andhra Pradesh,873721.106301,925839.118118,978581.462125,1.148471e+06,1.303524e+06,1439673.87
1,Arunachal Pradesh,25334.870000,30023.650000,30525.350000,3.477476e+04,3.962996e+04,43991.61
2,Assam,309336.320000,346850.680000,339802.980000,4.114538e+05,4.931666e+05,565400.58
3,Bihar,527975.819958,581855.484686,567262.504733,6.503024e+05,7.513956e+05,854429.00
4,Chhattisgarh,327106.660000,344672.040000,352327.510000,4.105249e+05,4.643990e+05,505886.51


In [9]:
population.head()

,state/ut,2018,2019,2020,2021,2022,2023,2024,2025,2026
0,Jammu & Kashmir*(UT),13101,13203,13305,13408,13505,13603,13701,13798,13896
1,Himachal Pradesh,7253,7300,7347,7394,7431,7468,7505,7542,7579
2,Punjab,29619,29859,30099,30339,30535,30730,30926,31122,31318
3,Haryana,28266,28672,29077,29483,29846,30209,30573,30936,31299
4,NCT of Delhi*,19435,19814,20193,20571,20965,21359,21752,22146,22540


### **Initial Integrity Check - Obvious Errors**

```Bill Table```

In [10]:
bill.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3664 entries, 0 to 3663
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   state_code           3664 non-null   int64  
 1   state_name           3664 non-null   object 
 2   year                 3664 non-null   int64  
 3   month                3664 non-null   int64  
 4   intra_suplliers      3664 non-null   object 
 5   intra_bill           3664 non-null   object 
 6   intra_assets         3664 non-null   object 
 7   inter_out_suplliers  3664 non-null   int64  
 8   inter_out_bill       3664 non-null   int64  
 9   inter_out_asset      3664 non-null   float64
 10  inter_in_suplliers   3664 non-null   int64  
 11  inter_in_bill        3664 non-null   int64  
 12  inter_in_asset       3664 non-null   float64
dtypes: float64(2), int64(7), object(4)
memory usage: 372.3+ KB


In [11]:
bill.columns

Index(['state_code', 'state_name', 'year', 'month', 'intra_suplliers',
       'intra_bill', 'intra_assets', 'inter_out_suplliers', 'inter_out_bill',
       'inter_out_asset', 'inter_in_suplliers', 'inter_in_bill',
       'inter_in_asset'],
      dtype='object')

```Summary Of Numerical Columns```

In [12]:
actual_numeric = ['intra_suplliers',
       'intra_bill', 'intra_assets', 'inter_out_suplliers', 'inter_out_bill',
       'inter_out_asset', 'inter_in_suplliers', 'inter_in_bill',
       'inter_in_asset']


bill[actual_numeric].describe().round(2).T[['min','50%','max','std','mean']].rename({'50%':'median'},axis = 1)

,min,median,max,std,mean
inter_out_suplliers,0.00,6697.00,168901.00,37773.49,25316.82
inter_out_bill,0.00,195438.00,8649985.00,1336783.14,771196.64
inter_out_asset,0.00,9506.45,254765.99,39191.20,26511.34
inter_in_suplliers,5.00,24052.50,259235.00,56359.00,47376.21
inter_in_bill,20.00,356606.50,5399958.00,987534.15,768406.13
inter_in_asset,0.38,12866.70,259141.83,37841.94,27908.82


---
3 columns are showing minimum value as 0, might be an issue in the data, worth checking

---

`Checking rows where value is 0 in any columns`

In [13]:
zero_cols = [ 'inter_out_suplliers', 'inter_out_bill','inter_out_asset', ]

print(f"Rows with 0 as values in any column: \n{(bill[zero_cols]==0).sum()}")

print("Rows with 0 values: ")
bill[(bill[zero_cols]==0).any(axis=1)].sort_values(by = ['state_name','year','month'],ascending=True)

Rows with 0 as values in any column: 
inter_out_suplliers    45
inter_out_bill         45
inter_out_asset        47
dtype: int64
Rows with 0 values: 


,state_code,state_name,year,month,intra_suplliers,intra_bill,intra_assets,inter_out_suplliers,inter_out_bill,inter_out_asset,inter_in_suplliers,inter_in_bill,inter_in_asset
2366,25,DAMAN AND DIU,2021,10,0,0,0,0,0,0.0,10,22,0.383400
2362,25,DAMAN AND DIU,2022,2,0,0,0,0,0,0.0,19,60,2.086800
2375,25,DAMAN AND DIU,2023,1,0,0,0,0,0,0.0,20,25,0.988600
2396,25,DAMAN AND DIU,2023,4,0,0,0,0,0,0.0,11,20,0.596800
2395,25,DAMAN AND DIU,2023,5,0,0,0,0,0,0.0,13,20,0.849900
2394,25,DAMAN AND DIU,2023,6,0,0,0,0,0,0.0,21,33,2.185500
2393,25,DAMAN AND DIU,2023,7,0,0,0,0,0,0.0,15,28,1.435300
2388,25,DAMAN AND DIU,2023,12,0,0,0,0,0,0.0,25,46,1.515900
2387,25,DAMAN AND DIU,2024,1,0,0,0,0,0,0.0,24,29,0.770900
2386,25,DAMAN AND DIU,2024,2,0,0,0,0,0,0.0,23,39,1.582900


---
0's are only from lakshadeep and daman and diu, since daman diu merged with dadar and nagar haveli, I will treat those 2 states as same state from the year 2019, for the sake of this analysis, 

and technically - **no company would want to open a hub in island like lakshadeep or andamans either, I will remove those 2 islands as well**

---

`Adding Dadar and Daman Diu together from the year 2019 onwards`

In [14]:
bill['state_name'].unique()

array([' JAMMU AND KASHMIR', 'JAMMU AND KASHMIR', ' HIMACHAL PRADESH',
       'HIMACHAL PRADESH', ' PUNJAB', 'PUNJAB', ' CHANDIGARH',
       'CHANDIGARH', ' UTTARAKHAND', 'UTTARAKHAND', ' HARYANA', 'HARYANA',
       ' DELHI', 'DELHI', ' RAJASTHAN', 'RAJASTHAN', ' UTTAR PRADESH',
       'UTTAR PRADESH', ' BIHAR', 'BIHAR', ' SIKKIM', 'SIKKIM',
       ' ARUNACHAL PRADESH', 'ARUNACHAL PRADESH', ' NAGALAND', 'NAGALAND',
       ' MANIPUR', 'MANIPUR', ' MIZORAM', 'MIZORAM', ' TRIPURA',
       'TRIPURA', ' MEGHALAYA', 'MEGHALAYA', ' ASSAM', 'ASSAM',
       ' WEST BENGAL', 'WEST BENGAL', ' JHARKHAND', 'JHARKHAND',
       ' ODISHA', 'ODISHA', ' CHHATTISGARH', 'CHHATTISGARH',
       ' MADHYA PRADESH', 'MADHYA PRADESH', ' GUJARAT', 'GUJARAT',
       ' DAMAN AND DIU', 'DAMAN AND DIU', ' DADRA AND NAGAR HAVELI',
       'DADRA AND NAGAR HAVELI', ' MAHARASHTRA', 'MAHARASHTRA',
       ' KARNATAKA', 'KARNATAKA', ' GOA', 'GOA', ' LAKSHADWEEP',
       'LAKSHADWEEP', ' KERALA', 'KERALA', ' TAMIL NADU', 'TA

---
There are duplicacy in column names, with leading and trailling whitespaces, need them to be removed

---

`Removing spaces from state names`

In [15]:
bill['state_name'] = bill['state_name'].str.strip()

In [16]:
bill['state_name'].unique()

array(['JAMMU AND KASHMIR', 'HIMACHAL PRADESH', 'PUNJAB', 'CHANDIGARH',
       'UTTARAKHAND', 'HARYANA', 'DELHI', 'RAJASTHAN', 'UTTAR PRADESH',
       'BIHAR', 'SIKKIM', 'ARUNACHAL PRADESH', 'NAGALAND', 'MANIPUR',
       'MIZORAM', 'TRIPURA', 'MEGHALAYA', 'ASSAM', 'WEST BENGAL',
       'JHARKHAND', 'ODISHA', 'CHHATTISGARH', 'MADHYA PRADESH', 'GUJARAT',
       'DAMAN AND DIU', 'DADRA AND NAGAR HAVELI', 'MAHARASHTRA',
       'KARNATAKA', 'GOA', 'LAKSHADWEEP', 'KERALA', 'TAMIL NADU',
       'PUDUCHERRY', 'ANDAMAN AND NICOBAR', 'TELANGANA', 'ANDHRA PRADESH',
       'LADAKH', 'Other Territory'], dtype=object)

`Merging 2 states, to act as one`

In [17]:
bill.drop('state_code',axis=1,inplace=True)

In [18]:
states_to_merge = ['DADRA AND NAGAR HAVELI','DAMAN AND DIU']
new_name = 'DADRA NAGAR HAVELI & DAMAN DIU'

bill['state_name'] = bill['state_name'].replace(states_to_merge,new_name)


bill = bill.groupby(['state_name','year','month'],as_index=False)[actual_numeric].sum()

`Removing the islands and Other territory`

In [19]:
bill['state_name'].unique()

array(['ANDAMAN AND NICOBAR', 'ANDHRA PRADESH', 'ARUNACHAL PRADESH',
       'ASSAM', 'BIHAR', 'CHANDIGARH', 'CHHATTISGARH',
       'DADRA NAGAR HAVELI & DAMAN DIU', 'DELHI', 'GOA', 'GUJARAT',
       'HARYANA', 'HIMACHAL PRADESH', 'JAMMU AND KASHMIR', 'JHARKHAND',
       'KARNATAKA', 'KERALA', 'LADAKH', 'LAKSHADWEEP', 'MADHYA PRADESH',
       'MAHARASHTRA', 'MANIPUR', 'MEGHALAYA', 'MIZORAM', 'NAGALAND',
       'ODISHA', 'Other Territory', 'PUDUCHERRY', 'PUNJAB', 'RAJASTHAN',
       'SIKKIM', 'TAMIL NADU', 'TELANGANA', 'TRIPURA', 'UTTAR PRADESH',
       'UTTARAKHAND', 'WEST BENGAL'], dtype=object)

In [20]:
remove_list = ['LAKSHADWEEP', 'ANDAMAN AND NICOBAR', 'Other Territory']

bill = bill[~bill['state_name'].isin(remove_list)]

---
Removing and Merging of problematic states are done here

---

`Check the count of each state, how many months of values it have, and what is the minumum and maximum year for all`

In [21]:
bill.groupby('state_name')['month'].count().reset_index(name = 'counts')

,state_name,counts
0,ANDHRA PRADESH,97
1,ARUNACHAL PRADESH,97
2,ASSAM,97
3,BIHAR,97
4,CHANDIGARH,97
5,CHHATTISGARH,97
6,DADRA NAGAR HAVELI & DAMAN DIU,97
7,DELHI,97
8,GOA,97
9,GUJARAT,97


In [24]:
bill['date'] = pd.to_datetime(bill[['year', 'month']].assign(day=1))

bill.groupby('state_name',as_index=False)['date'].agg(['min', 'max', 'count'])

,state_name,min,max,count
0,ANDHRA PRADESH,2018-07-01,2026-07-01,97
1,ARUNACHAL PRADESH,2018-07-01,2026-07-01,97
2,ASSAM,2018-07-01,2026-07-01,97
3,BIHAR,2018-07-01,2026-07-01,97
4,CHANDIGARH,2018-07-01,2026-07-01,97
5,CHHATTISGARH,2018-07-01,2026-07-01,97
6,DADRA NAGAR HAVELI & DAMAN DIU,2018-07-01,2026-07-01,97
7,DELHI,2018-07-01,2026-07-01,97
8,GOA,2018-07-01,2026-07-01,97
9,GUJARAT,2018-07-01,2026-07-01,97


---
Ladakh have less months of data than other months, and it is also remote location for a hub..safe to remove from the data

---

`Removing ladakh`

In [28]:
bill = bill[bill['state_name'] != 'LADAKH']

`Checking the stats after state cleaning`

In [29]:
actual_numeric = ['intra_suplliers',
       'intra_bill', 'intra_assets', 'inter_out_suplliers', 'inter_out_bill',
       'inter_out_asset', 'inter_in_suplliers', 'inter_in_bill',
       'inter_in_asset']


bill[actual_numeric].describe().round(2).T[['min','50%','max','std','mean']].rename({'50%':'median'},axis = 1)

,min,median,max,std,mean
inter_out_suplliers,16.00,8972.00,168901.00,39082.95,28973.57
inter_out_bill,47.00,277976.00,8649985.00,1395652.33,882636.52
inter_out_asset,2.94,15483.23,254765.99,40543.95,30321.54
inter_in_suplliers,429.00,30387.00,259235.00,57199.34,54159.10
inter_in_bill,3641.00,494553.00,5399958.00,1010096.71,878608.93
inter_in_asset,107.81,17083.62,259141.83,38933.85,31861.57


---
Problem of 0 values in data is fixed

---